# GLiNER encoder

> Route A: the encoder inside a GLiNER2 checkpoint (GLiNER2.5-Decide's DeBERTa-v3-large) under the sysone head, loaded with transformers alone, with the checkpoint's classifier as a warm start.

In [ ]:
#| export models
from sysone.template import RowTemplate as _RowTemplate

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import json, logging, tempfile, torch
import torch.nn as nn
from pathlib import Path
from safetensors.torch import save_file
from transformers import AutoConfig, AutoModel, AutoTokenizer, DebertaV2Config
from sysone.core import *
from sysone.template import RowTemplate, show_row
from sysone.models import *
logging.getLogger("transformers").setLevel(logging.ERROR)
import warnings; warnings.filterwarnings("ignore", message=r".*torch\.jit\.script", category=FutureWarning)  # DeBERTa-v2's modeling code on Python 3.14
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

[GLiNER2.5-Decide](https://huggingface.co/fastino/GLiNER2.5-Decide) (Apache-2.0) is already a slot-scoring decision model: GLiNER2's classification head is Laya's slot head with a learned anchor, `[L]`, in place of `[MASK]`. Its encoder can sit under the sysone head with no new dependency, which gives a decision-tuned encoder as the frozen backbone of the cached-features path. This notebook follows the encoder outline of the NeoMME notebook.

Two ways to use a GLiNER2 checkpoint, one notebook each:

```{mermaid}
flowchart TB
    CK["a GLiNER2 checkpoint<br/>(GLiNER2.5-Decide)"] --> Q{"what is it for?"}
    Q -->|"decisions, with sysone's cache, regimes,<br/>RLCD, calibration and export"| A["Route A, this notebook:<br/>its DeBERTa encoder under a DecisionHead,<br/>transformers only"]
    Q -->|"GLiNER2 as it is,<br/>its extraction features included"| B["Route B, the backend notebook:<br/>the gliner2 package behind a GlinerDecider"]
```

## 1. Load and inspect

A GLiNER2 checkpoint holds `config.json` (`model_type: extractor`, `architecture: span` or `boundary`), `encoder_config/config.json` (a `deberta-v2` config for Decide: 24 layers, width 1,024, relative attention with 256 buckets), the tokenizer (DeBERTa-v3's vocabulary plus ten schema tokens, `[SEP_STRUCT] [SEP_TEXT] [P] [C] [E] [R] [L] [EXAMPLE] [OUTPUT] [DESCRIPTION]`), and one `model.safetensors` with every tensor: the encoder under an `encoder.` prefix, the classifier MLP (`classifier.0`, width → 2 × width, ReLU, `classifier.2`, → 1; `classifier.3` on boundary checkpoints, which have a dropout in between), and span and count heads. A tiny checkpoint in the same layout, a two-layer DeBERTa-v2:

In [ ]:
GLINER_TOKENS = ["[SEP_STRUCT]", "[SEP_TEXT]", "[P]", "[C]", "[E]", "[R]", "[L]", "[EXAMPLE]", "[OUTPUT]", "[DESCRIPTION]"]

def tiny_gliner(path, architecture="span"):
    path = Path(path); path.mkdir(parents=True, exist_ok=True)
    tok = AutoTokenizer.from_pretrained("fixtures/tiny-modernbert")
    tok.add_special_tokens({"additional_special_tokens": GLINER_TOKENS})
    tok.save_pretrained(path)
    cfg = DebertaV2Config(vocab_size=len(tok), hidden_size=64, num_hidden_layers=2, num_attention_heads=2, intermediate_size=128,
                          max_position_embeddings=512, relative_attention=True, position_buckets=32, pos_att_type=["p2c", "c2p"],
                          share_att_key=True, norm_rel_ebd="layer_norm", position_biased_input=False, type_vocab_size=0,
                          pad_token_id=tok.pad_token_id)
    cfg.save_pretrained(path / "encoder_config")
    torch.manual_seed(0)
    enc = AutoModel.from_config(cfg, attn_implementation="eager")
    sd = {f"encoder.{k}": v for k, v in enc.state_dict().items()}
    H = cfg.hidden_size; last = "3" if architecture == "boundary" else "2"
    sd |= {"classifier.0.weight": torch.randn(2 * H, H) * 0.1, "classifier.0.bias": torch.zeros(2 * H),
           f"classifier.{last}.weight": torch.randn(1, 2 * H) * 0.1, f"classifier.{last}.bias": torch.zeros(1),
           "span_rep.span_rep_layer.out_project.0.weight": torch.randn(4, 4), "count_pred.0.weight": torch.randn(4, 4)}
    save_file({k: v.contiguous() for k, v in sd.items()}, str(path / "model.safetensors"))
    (path / "config.json").write_text(json.dumps({"model_type": "extractor", "architecture": architecture, "max_width": 8,
                                                  "counting_layer": "count_lstm", "model_name": "microsoft/deberta-v3-large"}))
    return path, enc, sd

ck, ref_enc, ref_sd = tiny_gliner(Path(tempfile.mkdtemp()) / "tiny-decide")
sorted(p.name for p in ck.iterdir()), sorted({k.split(".")[0] for k in ref_sd})

(['config.json',
  'encoder_config',
  'model.safetensors',
  'tokenizer.json',
  'tokenizer_config.json'],
 ['classifier', 'count_pred', 'encoder', 'span_rep'])

## 2. Loading the encoder

The encoder is built from `encoder_config` and filled from the tensors whose names end in its own parameter names, with the prefix discovered from the embedding matrix rather than assumed. The load is strict: every parameter of the encoder must land, including the resized vocabulary, and the span and count tensors are dropped. DeBERTa-v2 has no SDPA kernel, so it runs with eager attention. About fifty lines, and no `gliner2` import.

In [ ]:
#| export models
_EMBED_KEYS = ("embeddings.word_embeddings.weight", "embeddings.tok_embeddings.weight")

def is_gliner_checkpoint(src:str, revision:str|None=None) -> bool:
    "Whether `src` (a folder or Hub repo) is a GLiNER2 checkpoint: an extractor config and an encoder_config folder"
    p = Path(src)
    if p.exists():
        if not ((p / "config.json").exists() and (p / "encoder_config" / "config.json").exists()): return False
        c = json.loads((p / "config.json").read_text())
    else:
        if str(src).startswith(("/", ".")): return False
        try:
            from huggingface_hub import hf_hub_download, file_exists
            if not file_exists(src, "encoder_config/config.json", revision=revision): return False
            c = json.loads(Path(hf_hub_download(src, "config.json", revision=revision)).read_text())
        except Exception: return False
    return c.get("model_type") == "extractor" or "architecture" in c

def _gliner_file(src, name, revision=None) -> Path:
    p = Path(src) / name
    if p.exists(): return p
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(src, name, revision=revision))

def _gliner_dir(src, revision=None) -> Path:
    if Path(src).exists(): return Path(src)
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(src, revision=revision, allow_patterns=["config.json", "encoder_config/*", "model.safetensors",
                                                                          "tokenizer*", "special_tokens_map.json", "added_tokens.json", "*.model"]))

def load_gliner_encoder(src:str, revision:str|None=None, dtype=None):
    "The encoder of a GLiNER2 checkpoint, built from `encoder_config` and loaded strictly from `model.safetensors`"
    from safetensors import safe_open
    d = _gliner_dir(src, revision)
    cfg = AutoConfig.from_pretrained(d / "encoder_config")
    kw = {"attn_implementation": "eager"} if cfg.model_type in ("deberta-v2", "deberta") else {}
    enc = AutoModel.from_config(cfg, **kw)
    with safe_open(str(d / "model.safetensors"), "pt") as f:
        names = list(f.keys())
        prefix = next((n[:-len(k)] for n in names for k in _EMBED_KEYS if n.endswith(k)), None)
        if prefix is None: raise ValueError(f"{src}: no embedding matrix found among the checkpoint's tensors")
        sd = {n[len(prefix):]: f.get_tensor(n) for n in names if n.startswith(prefix)}
    missing, unexpected = enc.load_state_dict(sd, strict=False)
    if missing or unexpected:
        raise ValueError(f"{src}: partial encoder load (prefix {prefix!r}): missing {missing[:3]}, unexpected {unexpected[:3]}")
    enc = enc.float()
    return enc.to(dtype) if dtype is not None else enc

In [ ]:
enc = load_gliner_encoder(str(ck))
test_eq(all(torch.equal(enc.state_dict()[k], ref_enc.state_dict()[k]) for k in ref_enc.state_dict()), True)
test_eq(is_gliner_checkpoint(str(ck)), True)
test_eq(is_gliner_checkpoint("fixtures/tiny-modernbert"), False)

A checkpoint that lacks part of the encoder is refused, rather than loaded with random layers:

In [ ]:
from safetensors.torch import load_file

In [ ]:
bad = Path(tempfile.mkdtemp()) / "bad"
tiny_gliner(bad)
t = load_file(str(bad / "model.safetensors")); t.pop("encoder.encoder.layer.1.output.dense.weight")
save_file(t, str(bad / "model.safetensors"))
test_fail(lambda: load_gliner_encoder(str(bad)), contains="partial encoder load")

## 3. The template

The `gliner` preset mirrors the layout GLiNER2 trains on, `( [P] task: prompt ( [L] label … ) ) [SEP_TEXT] text`, with the question type as the task and `[L]` as the anchor. GLiNER2's processor adds no `[CLS]` or `[SEP]` around it (it tokenizes each word, label and bracket on its own), so the preset has no start or end token; this differs from an earlier sketch of the layout with `[CLS]` and `[SEP]`. GLiNER2 also lowercases the text and ends it with a period; `Lower(instructions=False, options=False)` reproduces the first if wanted. The `laya` template works unchanged on the same tokenizer too, with DeBERTa's own `[CLS]`, `[SEP]` and `[MASK]`; since the public DeBERTa-v3 weights are a discriminator without a masked-LM head, `[MASK]` starts weaker there than `[L]`, which was trained on GLiNER2's data and the decision post-training, so `gliner` is the one to try first.

In [ ]:
RowTemplate.preset("gliner")

RowTemplate(gliner: '{start}( [P] {type}: {instructions} ( {options} ) ){sep}{state}{end}', option='{mask} {text}', mask='[L]', budgets 512/192/48)

## 4. The spec

`EncoderSpec.from_pretrained` recognises a GLiNER2 checkpoint through `ENCODER_SOURCES`: the config is the encoder's, the tokenizer is the checkpoint's, the template is `gliner`, and the loader is the strict one above.

In [ ]:
#| export models
def _gliner_spec(id:str, revision:str|None=None, **kw) -> EncoderSpec:
    d = _gliner_dir(id, revision)
    cfg, tok = AutoConfig.from_pretrained(d / "encoder_config"), AutoTokenizer.from_pretrained(d)
    t = _RowTemplate.preset("gliner")
    if t.max_len > getattr(cfg, "max_position_embeddings", t.max_len): t = t.replace(max_len=cfg.max_position_embeddings)
    return EncoderSpec(id, cfg, tok, revision=revision, source="gliner2-checkpoint", template=t,
                       loader=lambda spec, dtype=None, **kw: load_gliner_encoder(id, revision, dtype))

ENCODER_SOURCES.insert(0, ("gliner2", is_gliner_checkpoint, _gliner_spec))

In [ ]:
spec = EncoderSpec.from_pretrained(str(ck))
spec

EncoderSpec(/var/folders/ym/y7_gxdb90s312g8bz06h3qqm0000gn/T/tmp5d14f0_f/tiny-decide: deberta-v2, width 64, context 512, text, template gliner)

In [ ]:
test_eq((spec.source, spec.family, spec.template.name, spec.width), ("gliner2-checkpoint", "deberta-v2", "gliner", 64))
b = spec.builder()
row = b.build("Duplicate charge on invoice #4411", {"type": "choice", "instructions": "Which department should handle this request?",
              "criteria": {"billing": "invoices, payments, refunds", "technical": "bugs, outages"}})
print(show_row(row, spec.tokenizer))
test_eq([row.ids[m] for m in row.markers], [spec.tokenizer.convert_tokens_to_ids("[L]")] * 2)
test_eq(row.ids[0] == spec.tokenizer.cls_token_id, False)

( [P] choice: Which department should handle this request? ( [L] billing: invoices, payments, refunds [L] technical: bugs, outages ) ) [SEP_TEXT] Duplicate charge on invoice #4411
markers [21, 32] · qtype choice · 58 tokens


## 5. One forward pass, and the classifier as a warm start

The checkpoint's classifier is an MLP on each `[L]` vector; `DecisionHead(width, layers=0, scorer="gliner2", init_from=checkpoint)` copies it into the scorer and zeroes the type embedding, so a head-only sysone model on the `gliner` template scores options exactly as the checkpoint's classifier scores its labels, softmaxed across the row as GLiNER2's single-label heads are, before any training; fine-tuning then starts there rather than from a random head. Laya's head remains the default for every other encoder.

In [ ]:
#| export models
def _gliner_head(head:DecisionHead, src:str):
    "Copy a GLiNER2 checkpoint's classifier MLP into a `gliner2` scorer"
    if head.scorer_kind != "gliner2": raise ValueError("a GLiNER2 warm start needs DecisionHead(..., scorer='gliner2')")
    t = read_tensors(str(_gliner_dir(src)) if Path(src).exists() else src, "model.safetensors", select=lambda k: k.startswith("classifier."))
    last = max(int(k.split(".")[1]) for k in t)          # 2 on span checkpoints, 3 on boundary ones (a dropout before it)
    sd = {"0.weight": t["classifier.0.weight"], "0.bias": t["classifier.0.bias"],
          "2.weight": t[f"classifier.{last}.weight"], "2.bias": t[f"classifier.{last}.bias"]}
    head.scorer.load_state_dict({k: v.float() for k, v in sd.items()})
    with torch.no_grad(): head.type_emb.weight.zero_()
    return head

HEAD_SOURCES.insert(0, (is_gliner_checkpoint, _gliner_head))

In [ ]:
head = DecisionHead(spec.width, layers=0, scorer="gliner2", init_from=str(ck))
model = EncoderDecisionModel(spec.load_encoder(), head, spec).eval()
batch = collate([row], spec.tokenizer.pad_token_id)
with torch.no_grad():
    ours = model(**{k: v for k, v in batch.items() if k not in ("target", "label")})
    h = ref_enc.eval()(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state[0, row.markers]
    clf = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 1))
    clf.load_state_dict({"0.weight": ref_sd["classifier.0.weight"], "0.bias": ref_sd["classifier.0.bias"],
                         "2.weight": ref_sd["classifier.2.weight"], "2.bias": ref_sd["classifier.2.bias"]})
    test_close(ours[0], clf(h).squeeze(-1), eps=1e-5)
ours.softmax(-1)

tensor([[0.5000, 0.5000]])

Boundary checkpoints (GLiNER2.5-multi-Decide) keep their last classifier layer at index 3; the warm start maps it:

In [ ]:
ckb, _, sdb = tiny_gliner(Path(tempfile.mkdtemp()) / "tiny-boundary", architecture="boundary")
hb = DecisionHead(64, layers=0, scorer="gliner2", init_from=str(ckb))
test_eq(torch.equal(hb.scorer[2].weight, sdb["classifier.3.weight"]), True)
test_fail(lambda: DecisionHead(64, layers=0, init_from=str(ckb)), contains="scorer='gliner2'")

## 6. A head-only run

Route A is the text application with a GLiNER2 checkpoint as the encoder: the cache, every regime, RLCD, calibration and the Laya-shaped export all apply (the Laya format itself is written only for ModernBERT-family encoders, which Laya's runtimes load).

In [ ]:
from sysone.data import TypedDecisions
from sysone.learner import decision_learner

In [ ]:
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0)
learn = decision_learner(data, str(ck), train="head", cache="masks", head="gliner2", init_from=str(ck), preset="cpu",
                         loss="soft_ce", cache_dir=tempfile.mkdtemp())
learn.fit(2, lr=1e-3)
learn.show_batch(1)

{'loss': '1.215', 'grad_norm': '0.0003907', 'learning_rate': '0.001', 'epoch': '1'}
{'eval_loss': '1.191', 'eval_accuracy': '0.24', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.08402', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0', 'eval_accuracy_score': '0.2', 'eval_accuracy_noul': '0.5', 'eval_runtime': '0.0079', 'eval_samples_per_second': '3154', 'eval_steps_per_second': '252.3', 'epoch': '1'}
{'loss': '1.215', 'grad_norm': '0.0003254', 'learning_rate': '0.001', 'epoch': '2'}
{'eval_loss': '1.191', 'eval_accuracy': '0.12', 'eval_soft_accuracy': '0.3767', 'eval_brier': '0.26', 'eval_kl': '0.4434', 'eval_tv': '0.366', 'eval_ece': '0.204', 'eval_score_mae': '0.838', 'eval_within_one': '0.6', 'eval_n': 25, 'eval_accuracy_choice': '0', 'eval_accuracy_score': '0.1', 'eval_accuracy_noul': '0.25', 'eval_runtime': '0.0063', 'eval_samples_per_second': '3968', 'eval_steps_per_secon

In [ ]:
test_eq((learn.model.head.scorer_kind, learn.data.builder.template.name), ("gliner2", "gliner"))

In [ ]:
#| eval: false
data  = TypedDecisions.from_hub("LocalLLaMA/typed-decisions", valid="test")
learn = decision_learner(data, "fastino/GLiNER2.5-Decide", train="head", cache="masks", head="gliner2",
                         init_from="fastino/GLiNER2.5-Decide")                 # template "gliner": [L] as the anchor
learn.validate()                                                              # the checkpoint's classifier, before training
learn.fit_one_cycle(4, lr=1e-4)
learn.calibrate()
learn.export("decide-head")

## 7. What to watch

- **Precision.** DeBERTa-v3-large's attention overflows in fp16; train in bf16 or fp32. The cached path runs the encoder once, so fp32 on a laptop costs little.
- **Tokenization.** Decide's `tokenizer.json` was re-saved by transformers 5 with an NFC normaliser, where DeBERTa-v3's own is NFKC; ASCII text is unaffected, ligatures and full-width characters are not. `unicodedata.normalize("NFKC", text)` in a transform is the defensive choice.
- **Not carried over.** GLiNER2's word-level `max_len` and chunking, its span and count heads, and its constraint decoder: Route A is decisions only. Its extraction features stay in `gliner2`; see the GLiNER backend notebook.
- **Decide-1B** is a ModernBERT (`ettin-enc-from-dec-1b`, width 1,792), not a DeBERTa: its embedding matrix is `tok_embeddings`, which the prefix discovery handles, and its attention runs on SDPA.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()